In [1]:
POSE_TAG = "pose5"
JSON_FILE_PATH = f"./json/random/{POSE_TAG}.json"
OUT_FOLDER_PATH = f"./output/center/random/angle_right/{POSE_TAG}/"

In [2]:
from src.json_to_pose.base import Pose, PoseAnalyzer
from src.json_to_pose.values import blazepose_lines
from src.utils.process_data import transYAxis, scaleYAxis
import src.utils.plot_painter as plot_painter

import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# 參考類型
from mpl_toolkits.mplot3d import Axes3D

#### 初始設定

In [3]:
### 視角參數

### 預設
# elev, azim, roll = 20, 30, 0

## 正面影像
# elev, azim, roll = 30, 90, 0  # 正面
# elev, azim, roll = 30, 180, 0  # 右側

## 右側影像
# elev, azim, roll = 30, 0, 0  # 正面
# elev, azim, roll = 30, 270, 0  # 右側

In [4]:
# 顯示資料範圍
data_range_x = [-1, 1]  # x 軸
data_range_y = [-1, 1]  # y 軸
data_range_z = [0, 2]  # z 軸

# 圖表資訊顏色
c_left = "#f00"  # 左邊
c_center = "#000"  # 中間
c_right = "#00f"  # 右邊
c_dot = "#000"  # 關鍵點

is_3d = True  # 圖表是否為 3D
dot_size = 5  # 繪製關鍵點大小
line_lst = blazepose_lines  # 設定連線列表

In [5]:
# 資料物件
pose = Pose(JSON_FILE_PATH)
analyzer = PoseAnalyzer(pose)
out_path = Path(OUT_FOLDER_PATH)

# 所需資料
pose_kpt_positions = pose.get_all_pose_kpt_positions(is_3d)
line_positions = analyzer.get_all_line_positions(blazepose_lines, is_3d)
labels = analyzer.get_all_lhc_labels(is_3d)

In [6]:
# 建立資料夾
if not out_path.exists():
    out_path.mkdir(parents=True)

#### 右邊圖片輸出

In [7]:
elev, azim, roll = 30, 270, 0  # 右邊視角
output_file_prefix = out_path / POSE_TAG  # 輸出檔案前綴

In [8]:
for i in range(pose.get_image_count()):

    fig = plt.figure()  # 建立圖表基底圖片
    title = ""  # 標題名稱

    # 建立圖表座標(2D 或 3D)
    if not is_3d:
        ax = fig.add_subplot()
    else:
        ax: Axes3D = fig.add_subplot(projection="3d")
        ax.view_init(elev, azim, roll)

    if pose.get_pose_count(i) > 0:  # 如果圖片中有姿勢
        title = f"label: {analyzer.get_pose_lhc_label(i, 0, is_3d)}\nscore: {pose.get_pose_score(i, 0)}"

        # 取得資料
        p_dot = pose_kpt_positions[i]
        _, p_center, _ = line_positions[i]

        # 將座標進行反轉和平移
        add_y = np.array(p_dot)[:, 1].max()  # 取得要平移的 y 軸數值
        p_center = transYAxis(scaleYAxis(p_center, -1), add_y)

        # 設定肩膀和腰部線條資訊
        shoulder_line = np.array(p_center[1])
        hip_line = np.array(p_center[2])

        # 圖表繪製
        ax.plot(shoulder_line[:,0], shoulder_line[:,2], shoulder_line[:,1], label="shoulder")
        ax.plot(hip_line[:,0], hip_line[:,2], hip_line[:,1], label="hip")
        ax.legend()
    else:  # 如果圖片中沒有姿勢
        title = "No Pose"

    # 圖表設定
    ax.set_title(title)
    plot_painter.set_data_range(data_range_x, data_range_y, data_range_z, ax)

    # 儲存圖表
    plt.savefig(f"{output_file_prefix}({i:03d}).png")
    plt.close()  # 關閉圖表

#### 完成程式

In [9]:
print("Finish")

Finish
